The goal is to crawl in lexicographical order over the folders (and, within a folder, a lexicographical crawl over the files) that're under 'true_false_datasets', creating a data structure that implicitly assigns a globally (within-project) unique numeric id to each record in each dataset file.

In [38]:
import pandas as pd

from phi_3_5_constants import dsets_folder, dsets_index_path

true_false_datasets/categ_dataset_pairs.csv
true_false_datasets\animal_class/animal_class.csv
true_false_datasets\animal_class/animal_class_conj.csv
true_false_datasets\animal_class/animal_class_disj.csv
true_false_datasets\animal_class/neg_animal_class.csv
true_false_datasets\cities/cities.csv
true_false_datasets\cities/cities_conj.csv
true_false_datasets\cities/cities_disj.csv
true_false_datasets\cities/neg_cities.csv
true_false_datasets\element_symb/element_symb.csv
true_false_datasets\element_symb/element_symb_conj.csv
true_false_datasets\element_symb/element_symb_disj.csv
true_false_datasets\element_symb/neg_element_symb.csv
true_false_datasets\facts/facts.csv
true_false_datasets\facts/facts_conj.csv
true_false_datasets\facts/facts_disj.csv
true_false_datasets\facts/neg_facts.csv
true_false_datasets\inventors/inventors.csv
true_false_datasets\inventors/inventors_conj.csv
true_false_datasets\inventors/inventors_disj.csv
true_false_datasets\inventors/neg_inventors.csv
true_false_dat

In [3]:
dset_folder_file_names: list[tuple[str, str]] = []

categ_dirs = sorted([c_dir for c_dir in dsets_folder.iterdir() if c_dir.is_dir()])

for categ_dir in categ_dirs:
    dset_file_names_in_categ: list[str] = []
    for data_file in categ_dir.iterdir():
        if data_file.is_dir():
            print(f"WARNING- entry {data_file.name} in folder {categ_dir.name} is a folder rather than the expected data file")
        else:
            dset_file_names_in_categ.append(data_file.name)
            
    dset_file_names_in_categ = sorted(dset_file_names_in_categ)
    dset_folder_file_names.extend([(categ_dir.name, file_name_in_categ) for file_name_in_categ in dset_file_names_in_categ])
    

In [4]:
# dset_folder_file_names

In [22]:
dset_folder_file_names_df = pd.DataFrame.from_records(dset_folder_file_names, columns=["Categ_Folder", "Dataset_File"])

In [23]:
dset_folder_file_names_df

,Categ_Folder,Dataset_File
0,animal_class,animal_class.csv
1,animal_class,animal_class_conj.csv
2,animal_class,animal_class_disj.csv
3,animal_class,neg_animal_class.csv
4,cities,cities.csv
5,cities,cities_conj.csv
6,cities,cities_disj.csv
7,cities,neg_cities.csv
8,element_symb,element_symb.csv
9,element_symb,element_symb_conj.csv


In [24]:
#dset_folder_file_names_df.to_csv(dsets_folder / "categ_dataset_pairs.csv", index_label="Idx")

In [27]:
dset_folder_file_names_df['min_global_record_idx'] = -1
dset_folder_file_names_df['max_global_record_idx'] = -1

In [29]:
dset_folder_file_names_df.at[10, "Categ_Folder"]

'element_symb'

In [30]:
num_records_so_far = 0
for (idx, categ_folder_nm, dset_file_nm, _, _) in dset_folder_file_names_df.itertuples():
    curr_dset_df = pd.read_csv(dsets_folder / categ_folder_nm / dset_file_nm)
    dset_folder_file_names_df.at[idx, 'min_global_record_idx'] = num_records_so_far
    num_records_so_far += curr_dset_df.shape[0]
    dset_folder_file_names_df.at[idx, 'max_global_record_idx'] = num_records_so_far

In [31]:
dset_folder_file_names_df

,Categ_Folder,Dataset_File,min_global_record_idx,max_global_record_idx
0,animal_class,animal_class.csv,0,164
1,animal_class,animal_class_conj.csv,164,664
2,animal_class,animal_class_disj.csv,664,1164
3,animal_class,neg_animal_class.csv,1164,1328
4,cities,cities.csv,1328,2824
5,cities,cities_conj.csv,2824,4322
6,cities,cities_disj.csv,4322,4822
7,cities,neg_cities.csv,4822,6318
8,element_symb,element_symb.csv,6318,6504
9,element_symb,element_symb_conj.csv,6504,7004


In [32]:
dset_folder_file_names_df.to_csv(dsets_folder / "categ_dataset_pairs.csv", index_label="Idx")

In [41]:
dset_folder_file_names_df = pd.read_csv(dsets_folder / "categ_dataset_pairs.csv", index_col="Idx")
dset_folder_file_names_df

,Categ_Folder,Dataset_File,min_global_record_idx,max_global_record_idx
Idx,,,,
0,animal_class,animal_class.csv,0,164
1,animal_class,animal_class_conj.csv,164,664
2,animal_class,animal_class_disj.csv,664,1164
3,animal_class,neg_animal_class.csv,1164,1328
4,cities,cities.csv,1328,2824
5,cities,cities_conj.csv,2824,4322
6,cities,cities_disj.csv,4322,4822
7,cities,neg_cities.csv,4822,6318
8,element_symb,element_symb.csv,6318,6504


In [44]:
for (idx, categ_folder_nm, dset_file_nm, _, _) in dset_folder_file_names_df.itertuples():
    curr_dset_df = pd.read_csv(dsets_folder / categ_folder_nm / dset_file_nm)
    if 'statement' not in curr_dset_df.columns or 'label' not in curr_dset_df.columns:
        print(f"WARNING- in category {categ_folder_nm}, dataset {dset_file_nm} was read with column names that're inconsistent with the other datasets: {curr_dset_df.columns}")
    else:
        for index, row in curr_dset_df.iterrows():
            if row['statement'][-1] not in [".", "!", "?"]:
                print(f"WARNING- in category {categ_folder_nm}, dataset {dset_file_nm}'s {index}th row has value with no period at the end: {row['statement']}")

In [49]:
dset_folder_file_names_df["is_negated"] = False
dset_folder_file_names_df["is_conj"] = False
dset_folder_file_names_df["is_disj"] = False
#"is_other" refers to the "other" types of categories, the ones that don't have the simple "positive-negative-conjunction-disjunction" structure of dataset variants 
dset_folder_file_names_df["is_other"] = False

In [50]:
for (idx, categ_folder_nm, dset_file_nm, _,_,_,_,_,_) in dset_folder_file_names_df.itertuples():
    if categ_folder_nm in ["real_world_scenarios", "relative_comparison", "true_false"]:
        dset_folder_file_names_df.at[idx, 'is_other'] = True
    elif dset_file_nm.startswith("neg_"):
        dset_folder_file_names_df.at[idx, 'is_negated'] = True
    elif dset_file_nm.endswith("_conj.csv"):
        dset_folder_file_names_df.at[idx, 'is_conj'] = True
    elif dset_file_nm.endswith("_disj.csv"):
        dset_folder_file_names_df.at[idx, 'is_disj'] = True


In [51]:
dset_folder_file_names_df

,Categ_Folder,Dataset_File,min_global_record_idx,max_global_record_idx,is_negated,is_conj,is_disj,is_other
Idx,,,,,,,,
0,animal_class,animal_class.csv,0,164,False,False,False,False
1,animal_class,animal_class_conj.csv,164,664,False,True,False,False
2,animal_class,animal_class_disj.csv,664,1164,False,False,True,False
3,animal_class,neg_animal_class.csv,1164,1328,True,False,False,False
4,cities,cities.csv,1328,2824,False,False,False,False
5,cities,cities_conj.csv,2824,4322,False,True,False,False
6,cities,cities_disj.csv,4322,4822,False,False,True,False
7,cities,neg_cities.csv,4822,6318,True,False,False,False
8,element_symb,element_symb.csv,6318,6504,False,False,False,False


In [52]:
dset_folder_file_names_df.to_csv(dsets_index_path, index_label="Idx")